# Multivariate EDA

## Imports

In [ ]:
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import LabelEncoder

## Data Preparation

In [ ]:
data_df = pd.read_csv("../data/sample.csv")
column_nature_df = pd.read_csv("../data/column_nature.sample.csv")
categorical_features_df = column_nature_df[column_nature_df["columnNature"] == "categorical"]
categorical_features = categorical_features_df["columnName"].to_list()

label_encoder = LabelEncoder()
encoded_data_df = data_df.copy()

non_singular_columns = []
for column in encoded_data_df.columns:
    if encoded_data_df[column].value_counts().size > 1:
        non_singular_columns.append(column)

encoded_data_df = encoded_data_df.filter(non_singular_columns)
encoded_alive_data_df = data_df.copy().filter(non_singular_columns)
encoded_alive_data_df = encoded_alive_data_df[encoded_alive_data_df["Outcome"] == "Alive"]

ordinal_columns = ["BirthMultiplicity_term","BirthFinalModeDeliver_term"]

# encode categorical data
for feature in categorical_features:
    if feature in non_singular_columns:
        encoded_data_df[feature] = label_encoder.fit_transform(encoded_data_df[feature])
        encoded_alive_data_df[feature] = label_encoder.fit_transform(encoded_alive_data_df[feature])

## All Data

### Compute and Visualise Correlation Matrix

In [ ]:
correlation_df = encoded_data_df.corr(method="spearman", numeric_only=True)
display(correlation_df)

fig = px.imshow(round(correlation_df, 5), text_auto=True, aspect="auto")
fig.show()

fig, ax = plt.subplots(figsize=(20, 20))
sns.heatmap(correlation_df, linewidths=0.5, ax=ax)

In [ ]:
los_series = correlation_df["DurationHospStayDay"]
print(los_series.quantile(0.80))
print(los_series.quantile(0.20))
pd.DataFrame(los_series[(los_series > 0.3) | (los_series < -0.3)].sort_values(ascending=False))

In [ ]:
outcome_series = correlation_df["Outcome"]
print(outcome_series.quantile(0.80))
print(outcome_series.quantile(0.20))
pd.DataFrame(outcome_series[(outcome_series > 0.3) | (outcome_series < -0.3)].sort_values(ascending=False))

## Alive Data

### Compute and Visualise Correlation Matrix

In [ ]:
alive_correlation_df = encoded_alive_data_df.corr(method="spearman")
display(alive_correlation_df)

fig = px.imshow(round(alive_correlation_df, 5), text_auto=True, aspect="auto")
fig.show()

fig, ax = plt.subplots(figsize=(20, 20))
sns.heatmap(alive_correlation_df, linewidths=0.5, ax=ax)

### Alive only LOS correlation

In [ ]:
alive_los_series = alive_correlation_df["DurationHospStayDay"]
print(alive_los_series.quantile(0.80))
print(alive_los_series.quantile(0.20))

pd.DataFrame(alive_los_series[(alive_los_series > 0.3) | (alive_los_series < -0.3)].sort_values(ascending=False))